# 04 — How big is the train/serve skew, and does training on streaming features fix it?

Three versions of the same I_ML model (no look-ahead, 200 trees × 31 leaves), tested on
people it never saw, **on exactly the same frames**:

| | trained on | tested on | what it is |
|---|---|---|---|
| **(a)** | offline features | offline features | how we evaluated so far |
| **(b)** | offline features | streaming features | what the live system does **now** |
| **(c)** | streaming features | streaming features | **Phase 2: train on what you serve** |

`b − a` = the cost of the skew. `b − c` = what retraining wins back.
The curve (trained and tested on the streaming 2D angle) is the reference.
Finally the deployable model is retrained on REHAB24-6 streaming features.

In [ ]:
import os, sys, json, time, warnings
import numpy as np, pandas as pd
from scipy.stats import wilcoxon
import lightgbm as lgb

REPO  = r'F:\namal sir project\github repo\AI-based-Physiotherapy-project-'   # <-- your repo
FILES = {'rehab24': os.path.join(REPO, 'data', 'fit3d', 'feat_rehab24_stream.npz'),
         'fit3d':   os.path.join(REPO, 'data', 'fit3d', 'feat_fit3d_stream.npz')}
N_JOBS = max(1, (os.cpu_count() or 2) - 1)
sys.path[:0] = [REPO, os.path.join(REPO, 'scripts')]
from calibration import fit_curve, apply_calibration
from fit_calibration import extend_below
pd.set_option('display.width', 220)

def view_class(v):
    """camera-relative view (REHAB24-6 cam18 labels describe the subject's pose to cam17)"""
    v = str(v).lower(); cam18 = 'cam18' in v
    if 'half' in v: return 'half'
    if 'front' in v: return 'profile' if cam18 else 'front'
    if 'prof' in v or 'side' in v: return 'front' if cam18 else 'profile'
    return 'unknown'

frames = []
for name, path in FILES.items():
    if not os.path.exists(path):
        print('missing (skipped):', path); continue
    d = np.load(path, allow_pickle=False)
    names = [str(n) for n in d['feature_names']]
    off = pd.DataFrame(d['X_offline'], columns=names).add_prefix('off_')
    stm = pd.DataFrame(d['X_stream'], columns=names).add_prefix('st_')
    df = pd.concat([off, stm], axis=1)
    for k in ('y', 'person', 'recording', 'view', 'exercise', 'dataset'):
        df[k] = d[k]
    frames.append(df)
D = pd.concat(frames, ignore_index=True)
D['vclass'] = D['view'].map(view_class)
# the SAME frames for every condition: both feature versions have a 2D angle
D = D[np.isfinite(D['off_angle_2d']) & np.isfinite(D['st_angle_2d']) & np.isfinite(D['y'])].reset_index(drop=True)

DROP = {'bone_upper', 'bone_fore', 'sw_frac', 'is_right', 'std_angle', 'd_angle'}   # iml_nolook
FEATS = [n for n in names if n not in DROP]
print(D.groupby(['dataset', 'vclass']).agg(frames=('y', 'size'), people=('person', 'nunique'),
      recordings=('recording', 'nunique')))
print('features:', len(FEATS), FEATS)

## How different are the streaming features?

In [ ]:
drift = pd.DataFrame({f: [float(np.nanmedian(np.abs(D['off_' + f] - D['st_' + f]))),
                          float(np.nanpercentile(np.abs(D['off_' + f] - D['st_' + f]), 95))]
                      for f in FEATS}, index=['median |offline - stream|', 'p95']).T
drift['median offline value'] = [float(np.nanmedian(np.abs(D['off_' + f]))) for f in FEATS]
drift.round(3)
# features whose difference is large next to their typical value are where the skew comes from

## Train and test: (a), (b), (c) and the curve

In [ ]:
HP = dict(objective='l1', n_estimators=200, num_leaves=31, learning_rate=0.09, min_child_samples=80,
          subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, verbosity=-1, n_jobs=N_JOBS)

def fit(tr, prefix):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        return lgb.LGBMRegressor(**HP).fit(tr[[prefix + f for f in FEATS]].to_numpy(), tr['y'])

def pred(model, te, prefix):
    return np.clip(model.predict(te[[prefix + f for f in FEATS]].to_numpy()), 0, 180)

def curve_pred(tr, te):
    def one(a, b):
        for mc in (300, 100, 30):
            try:
                return extend_below(fit_curve(a, b, min_count=mc))
            except ValueError:
                pass
    pooled = one(tr['st_angle_2d'], tr['y']); out = np.empty(len(te))
    for vc, idx in te.groupby('vclass').indices.items():
        sub = tr[tr['vclass'] == vc]
        p = one(sub['st_angle_2d'], sub['y']) if len(sub) >= 2000 else None
        out[idx] = apply_calibration(te['st_angle_2d'].to_numpy(float)[idx], p or pooled)
    return out

def run(train_mask, test_mask, label):
    tr, te = D[train_mask], D[test_mask].reset_index(drop=True)
    m_off, m_st = fit(tr, 'off_'), fit(tr, 'st_')
    out = te[['dataset', 'person', 'recording', 'vclass', 'y']].copy()
    out['a_off_off'] = pred(m_off, te, 'off_')
    out['b_off_stream'] = pred(m_off, te, 'st_')
    out['c_stream_stream'] = pred(m_st, te, 'st_')
    out['curve_stream'] = curve_pred(tr, te)
    out['experiment'] = label
    return out

t0 = time.time()
is_fit, is_reh = D['dataset'] == 'fit3d', D['dataset'] == 'rehab24'
parts = []
if is_fit.any() and is_reh.any():
    parts += [run(is_reh, is_fit, 'E2a REHAB->Fit3D'), run(is_fit, is_reh, 'E2b Fit3D->REHAB')]
for p in sorted(D['person'].unique()):
    parts.append(run(D['person'] != p, D['person'] == p, 'E3 LOSO'))
    print('  held out', p, f'{time.time() - t0:.0f}s', flush=True)
P = pd.concat(parts, ignore_index=True)
print(f'finished in {(time.time() - t0) / 60:.1f} min')

## Results

In [ ]:
M = ['curve_stream', 'a_off_off', 'b_off_stream', 'c_stream_stream']
err = P[M].sub(P['y'], axis=0).abs()
rec = pd.concat([P[['experiment', 'dataset', 'person', 'recording', 'vclass']], err], axis=1) \
        .groupby(['experiment', 'dataset', 'person', 'recording', 'vclass'], as_index=False)[M].mean()

def table(r):
    t = r.groupby('experiment')[M].median().round(2)
    t['skew (b-a)'] = (t['b_off_stream'] - t['a_off_off']).round(2)
    t['fix (b-c)'] = (t['b_off_stream'] - t['c_stream_stream']).round(2)
    return t.assign(recordings=r.groupby('experiment').size())

print('ALL VIEWS - median per-recording MAE (deg)'); display(table(rec))
print('PROFILE VIEW (live setup)'); display(table(rec[rec.vclass == 'profile']))

def paired(r, label):
    s = r[r.experiment == 'E3 LOSO'].groupby('person')[['b_off_stream', 'c_stream_stream']].mean()
    if len(s) < 6:
        return None
    gain = s['b_off_stream'] - s['c_stream_stream']
    return dict(scope=label, people=len(s), median_gain=round(float(gain.median()), 2),
                better_for=f'{int((gain > 0).sum())}/{len(s)}', wilcoxon_p=round(float(wilcoxon(gain).pvalue), 4))

print('Does retraining on streaming features help? (E3, per person, b vs c)')
display(pd.DataFrame([x for x in (paired(rec, 'E3 LOSO'),
                                  paired(rec[rec.vclass == 'profile'], 'E3 LOSO profile')) if x]))

**How to read it**

* `skew (b−a)` close to 0 → the live features already match training; Phase 2 changes little.
* `skew (b−a)` of a degree or more → the live system is worse than the offline numbers say.
* `fix (b−c)` > 0 with a small p-value → training on streaming features recovers that loss: deploy the model below.
* Report (c) as the honest live accuracy from now on.

## Retrain the deployable model on REHAB24-6 streaming features

In [ ]:
reh = D[D['dataset'] == 'rehab24']
model = fit(reh, 'st_')
path = os.path.join(REPO, 'data', 'iml_nolook_rehab24_stream.txt')
model.booster_.save_model(path)
e3 = rec[rec.experiment == 'E3 LOSO']
meta = {'features': FEATS,                       # live IMLStream reads these names
        'params': {k: v for k, v in HP.items() if k != 'n_jobs'},
        'trained_on': 'REHAB24-6 only (CC BY-NC), streaming features',
        'features_built_by': 'iml_live.stream_features (frame by frame, causal)',
        'cv': {'E3_all_views_stream_deg': round(float(e3['c_stream_stream'].median()), 2),
               'E3_profile_stream_deg': round(float(e3[e3.vclass == 'profile']['c_stream_stream'].median()), 2)}}
json.dump(meta, open(path + '.json', 'w'), indent=1)
print('saved', path, os.path.getsize(path) // 1024, 'KB')
print('use it live:  python demo_cv.py --variant ML --iml-model data/iml_nolook_rehab24_stream.txt')

In [ ]:
# optional: record this run in MLflow (same database as mlflow_backfill.py)
try:
    import mlflow
    mlflow.set_tracking_uri('sqlite:///' + os.path.join(REPO, 'mlflow.db').replace('\\', '/'))
    mlflow.set_experiment('Physio-ROM-Estimation')
    with mlflow.start_run(run_name='phase2-streaming-retrain'):
        mlflow.set_tags({'stage': 'offline_evaluation', 'method': 'iml_nolook', 'phase': '2 - train/serve skew'})
        mlflow.log_params({**{k: v for k, v in HP.items() if k != 'n_jobs'}, 'features': len(FEATS)})
        for scope, r in (('all', rec), ('profile', rec[rec.vclass == 'profile'])):
            t = table(r)
            for exp, row in t.iterrows():
                for col in M + ['skew (b-a)', 'fix (b-c)']:
                    key = f"{scope}_{exp.split()[0]}_{col}".replace(' ', '').replace('(', '_').replace(')', '').replace('-', '_')
                    mlflow.log_metric(key, float(row[col]))
        mlflow.log_artifact(path, artifact_path='model'); mlflow.log_artifact(path + '.json', artifact_path='model')
    print('logged to MLflow')
except ImportError:
    print('mlflow not installed - skipped')